# Indian News Bias Analyzer

A live RSS-driven NLP + SQL notebook for clustering Indian news coverage, comparing sentiment and loaded-word usage across outlets, and exporting a daily bias dashboard.


In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

cwd = Path.cwd()
if (cwd / "common.py").exists():
    SRC_DIR = cwd
else:
    SRC_DIR = cwd / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from common import (  # noqa: E402
    FEED_SOURCES,
    LOADED_WORDS,
    OUTPUTS_DIR,
    REPORTS_DIR,
    article_keyword_counter,
    article_text,
    connect_db,
    count_loaded_words,
    ensure_directories,
    initialize_schema,
    today_iso_date,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 11

ensure_directories()
connection = connect_db()
initialize_schema(connection)
print("Database ready:", connection.execute("SELECT COUNT(*) AS count FROM articles").fetchone()["count"])


## 1. RSS Feed Ingestion & SQLite Setup

Install the live feed readers, ingest the four RSS sources, and persist the article stream into SQLite with URL deduplication.


In [ ]:
import time

import feedparser
import schedule

from common import FEED_SOURCES, REQUEST_HEADERS, article_text


def fetch_feed(source: str, url: str) -> list[dict[str, str | None]]:
    parsed = feedparser.parse(url, request_headers=REQUEST_HEADERS)
    records: list[dict[str, str | None]] = []
    for entry in parsed.entries:
        link = entry.get("link") or entry.get("id")
        if not link:
            continue
        records.append(
            {
                "source": source,
                "title": entry.get("title", "").strip(),
                "description": article_text(entry.get("summary") or entry.get("description"), None),
                "published_at": None,
                "url": link.strip(),
            }
        )
    return records


def fetch_source_records(source: str, urls: tuple[str, ...]) -> list[dict[str, str | None]]:
    last_records: list[dict[str, str | None]] = []
    for url in urls:
        records = fetch_feed(source, url)
        if records:
            return records
        last_records = records
    return last_records


def ingest_once() -> pd.DataFrame:
    initialize_schema(connection)
    inserted = 0
    for source, urls in FEED_SOURCES.items():
        for record in fetch_source_records(source, urls):
            cursor = connection.execute(
                """
                INSERT IGNORE INTO articles (source, title, description, published_at, url)
                VALUES (?, ?, ?, ?, ?)
                """,
                (record["source"], record["title"], record["description"], record["published_at"], record["url"]),
            )
            inserted += cursor.rowcount
    connection.commit()
    return pd.read_sql_query(
        "SELECT source, title, url, published_at FROM articles ORDER BY id DESC LIMIT 20",
        connection,
    )


def run_scheduler() -> None:
    ingest_once()
    schedule.every(6).hours.do(ingest_once)
    while True:
        schedule.run_pending()
        time.sleep(60)


latest_articles = ingest_once()
latest_articles.head()


## 2. Topic Clustering with TF-IDF & K-Means

Vectorize titles and descriptions, cluster articles into shared events, and write the cluster labels back into SQLite.


In [ ]:
def load_articles() -> pd.DataFrame:
    return pd.read_sql_query(
        "SELECT id, source, title, description, published_at, url, topic_cluster FROM articles ORDER BY id",
        connection,
    )


def cluster_articles(k: int = 15) -> tuple[pd.DataFrame, dict[int, str]]:
    articles = load_articles()
    articles["text"] = articles.apply(lambda row: article_text(row["title"], row["description"]), axis=1)
    articles = articles[articles["text"].str.len() > 0].copy()
    if len(articles) < 2:
        return articles, {}

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=4000, stop_words="english")
    matrix = vectorizer.fit_transform(articles["text"])
    cluster_count = min(max(2, int(len(articles) ** 0.5)), k, len(articles))
    model = KMeans(n_clusters=cluster_count, random_state=42, n_init="auto")
    articles["cluster_id"] = model.fit_predict(matrix)

    feature_names = vectorizer.get_feature_names_out()
    cluster_labels: dict[int, str] = {}
    for cluster_id, centroid in enumerate(model.cluster_centers_):
        top_terms = [feature_names[index].replace(" ", "-") for index in centroid.argsort()[-3:][::-1] if centroid[index] > 0]
        cluster_labels[cluster_id] = "-".join(top_terms) if top_terms else f"topic-{cluster_id + 1}"

    articles["topic_cluster"] = articles["cluster_id"].map(cluster_labels)

    for _, row in articles.iterrows():
        connection.execute("UPDATE articles SET topic_cluster = ? WHERE id = ?", (row["topic_cluster"], int(row["id"])))

    connection.commit()
    return articles, cluster_labels

clustered_articles, cluster_labels = cluster_articles(k=18)
clustered_articles[["source", "title", "topic_cluster"]].head(10)


## 3. Sentiment Analysis & Bias Scoring

Score each article with VADER, count loaded political words, and aggregate the bias metrics by source and topic.


In [ ]:
loaded_words = {
    "accused",
    "alleged",
    "angry",
    "blasted",
    "bombshell",
    "chaos",
    "collapsed",
    "controversial",
    "crisis",
    "damning",
    "demanded",
    "denounced",
    "disaster",
    "explosive",
    "fury",
    "hailed",
    "harsh",
    "horrific",
    "infuriated",
    "outrage",
    "praised",
    "ridiculed",
    "scandal",
    "slammed",
    "storm",
    "struck",
    "sweeping",
    "tumult",
    "warned",
    "wins",
    "woeful",
}

analyzer = SentimentIntensityAnalyzer()
articles = load_articles()
scored_rows = []

for _, row in articles.iterrows():
    text = article_text(row["title"], row["description"])
    if not text:
        continue
    sentiment = float(analyzer.polarity_scores(text)["compound"])
    tokens = [token.lower().strip(".,:;!?()[]{}\"'`") for token in text.split()]
    loaded_word_count = sum(token in loaded_words for token in tokens)
    loaded_word_ratio = loaded_word_count / max(len(tokens), 1)
    bias_score = abs(sentiment) * (1 + loaded_word_ratio)
    connection.execute(
        "UPDATE articles SET sentiment_score = ?, bias_score = ? WHERE id = ?",
        (sentiment, bias_score, int(row["id"])),
    )
    scored_rows.append(
        {
            "id": int(row["id"]),
            "source": row["source"],
            "topic": row.get("topic_cluster") or "unclustered",
            "sentiment_score": sentiment,
            "bias_score": bias_score,
            "loaded_word_count": loaded_word_count,
        }
    )

scores = pd.DataFrame(scored_rows)
summary = (
    scores.groupby(["source", "topic"], as_index=False)
    .agg(
        avg_sentiment=("sentiment_score", "mean"),
        avg_bias=("bias_score", "mean"),
        loaded_word_count=("loaded_word_count", "sum"),
        article_count=("id", "count"),
    )
)
summary["fetched_date"] = today_iso_date()

for _, row in summary.iterrows():
    connection.execute(
        """
        INSERT INTO bias_summary (source, topic, avg_sentiment, avg_bias, loaded_word_count, article_count, fetched_date)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ON DUPLICATE KEY UPDATE
          avg_sentiment = VALUES(avg_sentiment),
          avg_bias = VALUES(avg_bias),
          loaded_word_count = VALUES(loaded_word_count),
          article_count = VALUES(article_count)
        """,
        (row["source"], row["topic"], float(row["avg_sentiment"]), float(row["avg_bias"]), int(row["loaded_word_count"]), int(row["article_count"]), row["fetched_date"]),
    )

connection.commit()
summary.sort_values("avg_bias", ascending=False).head(10)


## 4. Framing Divergence & Keyword Analysis

Pull the top 10 keywords per source for the most important topics and compute keyword overlap to quantify framing differences.


In [ ]:
top_topics = (
    summary.groupby("topic", as_index=False)["article_count"].sum().sort_values("article_count", ascending=False).head(5)["topic"].tolist()
)

framing_rows = []
for topic in top_topics:
    topic_keywords = (
        pd.read_sql_query(
            """
            SELECT a.source, k.keyword, SUM(k.freq) AS freq
            FROM articles a
            JOIN keywords k ON a.id = k.article_id
            WHERE COALESCE(a.topic_cluster, 'unclustered') = ?
            GROUP BY a.source, k.keyword
            ORDER BY freq DESC, k.keyword ASC
            """,
            connection,
            params=(topic,),
        )
    )
    source_keywords = {
        source: topic_keywords.loc[topic_keywords["source"] == source, "keyword"].head(10).tolist()
        for source in topic_keywords["source"].unique()
    }
    overlap = {}
    for left, right in combinations(sorted(source_keywords), 2):
        left_set = set(source_keywords[left])
        right_set = set(source_keywords[right])
        union = left_set | right_set
        overlap.setdefault(left, {})[right] = len(left_set & right_set) / len(union) if union else 0.0
        overlap.setdefault(right, {})[left] = overlap[left][right]
    framing_rows.append({"topic": topic, "source_keywords": source_keywords, "keyword_overlap": overlap})

framing_rows[:2]
